# From URDF to a 1 kHz controller on a Raspberry Pi

**The jaxonomy → jaxterity → jaxility stack, end to end on real silicon.**

This tutorial takes one robot all the way from a model to a *signed, embedded
controller running on a Raspberry Pi 5 at 1 kHz* — and proves, step by step, that
what runs on the chip is bit-for-bit what you simulated.

The thesis is **"one model, one truth"**: there is a single source of truth for
the robot, and every artifact downstream — the dynamics, the controller, the
deployed binary, the attestation — is derived from it. Recalibrate the robot and
the *deployed binary* changes, with a cryptographic record to prove it.

The pipeline:

```
 jaxterity Robot  ──►  reduced params  ──►  JAX dynamics  ──►  acados controller
 (the source of truth)   (T-101)            f(x,u)            (LQR / MPC, jaxility)
        │                                                            │
        ▼                                                            ▼
  attestation handle ───────────────────────────────►  signed Arm binary  ──►  Pi 5 @ 1 kHz
                                                        + manifest (BLAKE3)      (HIL-verified)
```

> **Requirements.** The acados toolchain (`t_renderer`, `ACADOS_SOURCE_DIR`) and a
> host C compiler. The on-silicon section also needs a reachable Pi 5 named in
> `JAXILITY_HIL_SSH_HOST` with a native acados install; without it those cells
> degrade gracefully to the host result.

In [1]:
import os, platform, ctypes
from pathlib import Path

# acados ships dynamic libs; on macOS, SIP strips DYLD_LIBRARY_PATH from the
# Python subprocess, so preload them by absolute path (a no-op elsewhere).
def _preload_acados():
    root = os.environ.get("ACADOS_SOURCE_DIR")
    if not root or platform.system() != "Darwin":
        return
    for n in ("libblasfeo.dylib", "libhpipm.dylib", "libqpOASES_e.dylib", "libacados.dylib"):
        p = Path(root) / "lib" / n
        if p.exists():
            try: ctypes.CDLL(str(p))
            except OSError: pass
_preload_acados()

import jax, numpy as np
import jaxility, jaxterity
print("jaxterity", jaxterity.__version__ if hasattr(jaxterity, "__version__") else "(editable)")
print("jaxility ", jaxility.__version__ if hasattr(jaxility, "__version__") else "(editable)")
print("acados   ", "set" if os.environ.get("ACADOS_SOURCE_DIR") else "MISSING (host build cells will fail)")
print("Pi target", os.environ.get("JAXILITY_HIL_SSH_HOST", "(unset — on-silicon cells will fall back)"))

jaxterity 0.0.1.dev0
jaxility  0.0.1.dev0
acados    set
Pi target pi@raspberrypi.local


## Step 1 — The robot (jaxterity)

Everything starts with a `Robot`. We load the Cartpole from the zoo and stamp a
provenance record to model "calibration ran" — the point is that the robot has a
real **attestation handle**, a content hash of its calibrated identity. Every
artifact we build below is rooted on this handle.

In [2]:
import jaxterity.zoo as zoo

robot = zoo.load("cartpole").with_provenance(
    ("tutorial-recipe", "v0", "telemetry-hash"), calibrated=True
)
print("robot:", robot.name, "| calibration:", robot.calibration_state.name)
print("attestation_handle:", robot.attestation_handle[:32], "…")

robot: physics | calibration: CALIBRATED
attestation_handle: f7cf67f3b514149bc73391794f423997 …


## Step 2 — From the robot to dynamics

jaxterity exposes the robot's **reduced parameters** — the robot-faithful scalars
the closed-form dynamics need (`T-101`). The controller is built from *these*, not
from hand-typed constants, so calibrating a mass propagates all the way to the
binary. We turn them into a JAX function `f(state, control) → dstate`.

In [3]:
import jax.numpy as jnp
from jaxterity.zoo.cartpole import reduced_params

NX, NU = 4, 1
params = reduced_params(robot)
print("reduced params:", {k: round(float(v), 4) for k, v in params.items()})

def cartpole_dynamics(state, control):
    g, mp, mc, L = params["g"], params["mp"], params["mc"], params["L"]
    theta, x_dot, theta_dot = state[1], state[2], state[3]
    s, c = jnp.sin(theta), jnp.cos(theta)
    denom = mc + mp * s * s
    x_ddot = (control[0] + mp * s * (L * theta_dot**2 + g * c)) / denom
    theta_ddot = (-control[0]*c - mp*L*theta_dot**2*c*s - (mc+mp)*g*s) / (L*denom)
    return jnp.array([x_dot, theta_dot, x_ddot, theta_ddot])

reduced params: {'g': 9.81, 'mc': 1.0, 'mp': 0.1, 'L': 0.5}


## Step 3 — Compile a controller (jaxility)

Now jaxility's job: lower the JAX dynamics to **CasADi → acados**, wrap them in an
LQR template, and build a real, loadable controller. We build for the *host* first
(so we can validate before crossing to hardware). The result carries an
**attestation manifest** rooted on the robot's handle.

In [4]:
from jaxility.lowering import translate
from jaxility.templates import lqr
from jaxility.builder import build_for_target
from jaxility.targets import current_host_target
from jaxility.manifest import verify_manifest
import tempfile

INITIAL_STATE = (0.3, 0.0, 0.0, 0.0)
workdir = Path(tempfile.mkdtemp())

def build(robot, name):
    p = reduced_params(robot)
    def dyn(state, control):
        g, mp, mc, L = p["g"], p["mp"], p["mc"], p["L"]
        th, xd, thd = state[1], state[2], state[3]
        s, c = jnp.sin(th), jnp.cos(th); den = mc + mp*s*s
        return jnp.array([xd, thd,
            (control[0]+mp*s*(L*thd**2+g*c))/den,
            (-control[0]*c-mp*L*thd**2*c*s-(mc+mp)*g*s)/(L*den)])
    cf = translate(dyn, in_shapes=((NX,), (NU,)), name=name)
    spec = lqr(cf, Q=(10.,10.,1.,1.), R=(0.1,), initial_state=INITIAL_STATE,
               input_bounds=((-20.,),(20.,)), name=name, horizon_steps=20, time_horizon_s=1.0)
    bundle = build_for_target(dynamics=cf, spec=spec, target=current_host_target(),
        source_attestation_handle=bytes.fromhex(robot.attestation_handle),
        work_dir=workdir / name)
    return bundle, dyn   # hand back the JAX dynamics (the plant for the HIL reference)

bundle, jax_dyn = build(robot, "tut_cartpole")
print("artifact hash:", bundle.artifact.content_hash.hex()[:32], "…")
print("manifest verifies:", verify_manifest(bundle.manifest).ok)
print("rooted on the robot handle:",
      bundle.manifest.source_attestation_handle == bytes.fromhex(robot.attestation_handle))

artifact hash: 47e59c2cabc4008e56a65ed16de48de2 …
manifest verifies: True
rooted on the robot handle: True


## Step 4 — Does it match the simulator? (HIL parity)

The whole promise is *"what you simulate is what you ship."* We close the loop two
ways and compare them step-locked: a **host reference** (acados control + a JAX
RK4 plant) versus the **generated C binary** (acados control + acados-sim plant).
If they agree to ULP, the codegen is faithful.

In [5]:
from jaxility.hil import (CARTPOLE_LQR_SCHEMA, CARTPOLE_LQR_TRACE, LocalRunner,
    build_controller_hil_binary, generate_controller_hil_source, run_hil)
PLANT_DT = 1.0 / 20

def _rk4(dyn, x, u, dt):
    xj, uj = jnp.asarray(x), jnp.asarray(u)
    k1 = dyn(xj, uj); k2 = dyn(xj+0.5*dt*k1, uj)
    k3 = dyn(xj+0.5*dt*k2, uj); k4 = dyn(xj+dt*k3, uj)
    return np.asarray(xj + (dt/6.)*(k1+2*k2+2*k3+k4))

def host_reference(solver, dyn, n, seed=0):
    solver.reset(); x = np.array(INITIAL_STATE); x[0] += 0.01*seed
    pos, vel, tau = np.empty((n,2)), np.empty((n,2)), np.empty((n,1))
    for i in range(n):
        solver.set(0,"lbx",x); solver.set(0,"ubx",x); solver.solve()
        u = np.asarray(solver.get(0,"u")); pos[i],vel[i],tau[i] = x[:2],x[2:],u
        x = _rk4(dyn, x, u, PLANT_DT)
    return {"joint_position":pos, "joint_velocity":vel, "actuator_torque":tau}

gen = bundle.shared_library_path.parent
src = generate_controller_hil_source(model_name="tut_cartpole", nx=NX, nu=NU,
        trace=CARTPOLE_LQR_TRACE, initial_state=INITIAL_STATE)
exe = build_controller_hil_binary(generated_code_dir=gen, model_name="tut_cartpole",
        source=src, out_path=gen/"tut_cartpole_hil")
ref = host_reference(bundle.solver, jax_dyn, 40)
report = run_hil(ref, LocalRunner(exe), target_family="cortex-a76", dtype="float64",
        schema=CARTPOLE_LQR_SCHEMA, n_steps=40, seed=0)
report.assert_passed()
print("HIL parity: PASSED — compiled controller matches the simulator bit-for-bit")
worst = max(q.max_abs_error for q in report.equivalence.per_quantity)
print(f"worst-case divergence: {worst:.2e}  (within the cortex-a76 x float64 bound)")

HIL parity: PASSED — compiled controller matches the simulator bit-for-bit
worst-case divergence: 4.22e-15  (within the cortex-a76 x float64 bound)


## Step 5 — Is 1 kHz feasible? (solve-time)

A controller is only deployable if it fits inside the control period. We time the
per-cycle acados solve on the host as a feasibility indicator (the rigorous
number comes from the Pi, next).

In [6]:
from jaxility.bench import generate_controller_bench_source, run_controller_bench
bsrc = generate_controller_bench_source(model_name="tut_cartpole", nx=NX, nu=NU,
        initial_state=INITIAL_STATE, n_warmup=100)
bexe = build_controller_hil_binary(generated_code_dir=gen, model_name="tut_cartpole",
        source=bsrc, out_path=gen/"tut_cartpole_bench")
rec = run_controller_bench(LocalRunner(bexe), robot="cartpole", target_family="cortex-a76",
        target_name="host", n_cycles=2000, n_warmup=100)
s = rec.solve
print(f"host solve-time: mean {s.mean_ns/1e3:.1f}  p99 {s.p99_ns/1e3:.1f}  "
      f"max {s.max_ns/1e3:.1f} us  -> meets 1 kHz: {rec.meets_1khz}")

host solve-time: mean 32.1  p99 65.0  max 105.0 us  -> meets 1 kHz: True


## Step 6 — Cross to real silicon (the Pi 5)

This is the payoff. We ship the host-generated acados C to a tethered **Raspberry
Pi 5**, compile it natively on the Cortex-A76, and re-run the *same* HIL parity +
1 kHz benchmark — now on the hardware. (This is the LeKiwi's onboard Pi; set
`JAXILITY_HIL_SSH_HOST` to it. If unset/unreachable, the cell points you at the
setup — the host results above still hold.)

The cell below runs it live and prints the measured Cortex-A76 numbers: per-cycle
solve time around **~100 µs** — comfortably inside the 1 kHz (1000 µs) budget, with
roughly **7× worst-case headroom** — and on-Pi HIL parity matching the host
reference to ULP (the cross-compiled controller is bit-for-bit the simulator).

In [7]:
import subprocess
from jaxility.hil import build_controller_on_target

host = os.environ.get("JAXILITY_HIL_SSH_HOST")
if not host:
    print("JAXILITY_HIL_SSH_HOST unset — skipping the on-silicon run.")
    print("Set it to your Pi 5 (e.g. pi@raspberrypi.local) and re-run to deploy on hardware.")
else:
  try:
    acados = os.environ.get("JAXILITY_HIL_ACADOS") or subprocess.run(
        ["ssh","-o","BatchMode=yes","-o","ConnectTimeout=20",host,"echo $HOME/acados"],
        capture_output=True, text=True, timeout=30).stdout.strip()
    hil_runner = build_controller_on_target(host=host, generated_code_dir=gen,
        model_name="tut_cartpole",
        source=generate_controller_hil_source(model_name="tut_cartpole", nx=NX, nu=NU,
            trace=CARTPOLE_LQR_TRACE, initial_state=INITIAL_STATE),
        source_name="tut_cartpole_hil_main.c", remote_dir="/tmp/jx-tut-hil", remote_acados=acados)
    pi_report = run_hil(host_reference(bundle.solver, jax_dyn, 40), hil_runner,
        target_family="cortex-a76", dtype="float64", schema=CARTPOLE_LQR_SCHEMA, n_steps=40, seed=0)
    pi_report.assert_passed()
    print(f"on-Pi HIL parity ({host}): PASSED — controller runs on the Cortex-A76")

    bench_runner = build_controller_on_target(host=host, generated_code_dir=gen,
        model_name="tut_cartpole",
        source=generate_controller_bench_source(model_name="tut_cartpole", nx=NX, nu=NU,
            initial_state=INITIAL_STATE, n_warmup=100),
        source_name="tut_cartpole_bench_main.c", remote_dir="/tmp/jx-tut-bench", remote_acados=acados)
    pr = run_controller_bench(bench_runner, robot="cartpole", target_family="cortex-a76",
        target_name="pi5", n_cycles=5000, n_warmup=100)
    ps = pr.solve
    print(f"on-Pi solve-time (n={pr.n_cycles}): mean {ps.mean_ns/1e3:.1f}  "
          f"p99 {ps.p99_ns/1e3:.1f}  max {ps.max_ns/1e3:.1f} us -> meets 1 kHz: {pr.meets_1khz}")
  except Exception as e:
    print(f"on-silicon run skipped ({type(e).__name__}: {e}).")
    print("The Pi was unreachable this run; the host results above still hold.")

on-Pi HIL parity (pi@raspberrypi.local): PASSED — controller runs on the Cortex-A76


on-Pi solve-time (n=5000): mean 101.9  p99 138.9  max 157.1 us -> meets 1 kHz: True


## Step 7 — One model, one truth

The claim that makes the whole stack worth it: the deployed binary tracks the
*robot*, not a config file. We recalibrate the pole to **2× mass** and rebuild —
both the **attestation handle** and the **artifact hash** move. Nothing else
changed; the physics did, and the binary followed.

In [8]:
heavy = robot.with_parameters({"pole.mass": robot.parameters()["pole.mass"] * 2.0})
heavy_bundle, _ = build(heavy, "tut_cartpole_heavy")
print("handle moved:       ", robot.attestation_handle != heavy.attestation_handle)
print("artifact hash moved:", bundle.artifact.content_hash != heavy_bundle.artifact.content_hash)
print("\n-> recalibrate the robot, and the deployed binary changes — with proof.")

handle moved:        True
artifact hash moved: True

-> recalibrate the robot, and the deployed binary changes — with proof.


## Can I use my own robot — the LeKiwi?

Short answer: **your LeKiwi's Pi 5 is a first-class deploy target (you just ran on
it), but the LeKiwi *arm* is not yet a controlled plant in this lane.** The honest
breakdown:

| Role | Your LeKiwi | Why |
|---|---|---|
| **Deploy target** (the chip) | ✅ yes | Its Pi 5 is exactly the Cortex-A76 this tutorial cross-compiles for and ran on. |
| **Controlled plant** (the arm) | ⚠️ not yet | Two reasons below. |

1. **Dynamics don't lower to acados.** This lane lowers *closed-form* JAX dynamics
   (`f(x,u)`) through CasADi → acados over a **finite supported op subset**
   (tutorial #7). The articulated arm's dynamics come from **MJX**, whose
   contact/constraint solver reaches for operations *outside* that subset — so
   there's no acados model for the full arm. (The zoo has `cartpole`, `so100`,
   `unitree_g1`, `crazyflie`; there is no `lekiwi` dynamics model.)
2. **The bus isn't a 1 kHz torque loop.** The STS3215 servos are a **1 Mbps,
   position-controlled** bus, not a 1 kHz torque interface — so even a compiled
   controller couldn't close *this* loop on the physical servos at 1 kHz.

**The path that *does* make it "your robot":** derive a **reduced closed-form
model** from your arm's own parameters and lower *that*. The cell below loads the
SO-101/LeKiwi arm and pulls a link's mass/length — the seed for, say, a single-link
pendulum controller that is genuinely parameterised by your unit, compiles through
this exact pipeline, and runs on your Pi (validated in HIL against a simulated
plant). Closing the loop on the *physical* arm is a separate integration (a torque
interface), and is what the **calibration** and **predictive self-check** tutorials
build toward.

In [9]:
# Your LeKiwi's arm, imported — the starting point for a reduced model.
arm = zoo.load("so100")
movable = [j for j in arm.joints if j.joint_type.name not in ("FIXED", "FLOATING")]
print("so100 / LeKiwi arm:", len(movable), "movable joints:")
for j in movable[:6]:
    lim = j.limits
    print(f"  {j.name:14s}  range [{lim.lower:+.2f}, {lim.upper:+.2f}] rad")
print("\n-> these joint/link parameters are the seed for a closed-form reduced")
print("   model you could lower + deploy with the exact pipeline above.")

so100 / LeKiwi arm: 6 movable joints:
  gripper         range [-0.17, +1.75] rad
  wrist_roll      range [-2.74, +2.84] rad
  wrist_flex      range [-1.66, +1.66] rad
  elbow_flex      range [-1.69, +1.69] rad
  shoulder_lift   range [-1.75, +1.75] rad
  shoulder_pan    range [-1.92, +1.92] rad

-> these joint/link parameters are the seed for a closed-form reduced
   model you could lower + deploy with the exact pipeline above.


## Recap & where to go next

You took **one robot** from a model to a **signed controller running on a Pi 5 at
1 kHz**, and proved at every step that the chip matches the simulator. The deployed
binary is rooted on the robot's attestation handle — *one model, one truth*.

Next in the series:
- **Deploy a learned policy safely** — add an AI controller on top of this MPC with
  a guaranteed on-target fallback (dual-path), for the same ~1 kHz cost.
- **Sign and verify a deployment** — the BLAKE3 manifest, tamper-evidence, and what
  "covers both artifacts" means.
- **Calibrate a real robot from telemetry** — turn your LeKiwi's measured motion
  into the parameters this pipeline consumes (active sysid + the split capture/fit).